In [ ]:
# Imports and connection set up
import os, csv, time, pathlib
import psycopg
import pandas as pd
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")

from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

### TO CHANGE TO YOUR OWN POSTGRES PASSWORD
postgres_password = "ENTER YOUR OWN PASSWORD"

### Database name in my local postgres is ds6103_project, change it to your own if you have a different name
### All my sp are created with public schema i.e. public.town; public.hdb_block; change the corresponding stored procedures schema to your own if you have a different schema name
PG = dict(user="postgres", host="localhost", port=5432, db="ds6103_project")

URL = (f"postgresql+psycopg2://{PG['user']}:{quote_plus(postgres_password)}"
       f"@{PG['host']}:{PG['port']}/{PG['db']}")

engine = create_engine(URL)

with engine.connect() as conn:
    print(conn.execute(text("select version()")).scalar())

PostgreSQL 17.11 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit


In [5]:
# Getting file path for cleaned csv files
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    BASE_DIR = Path.cwd().parent 

CLEANED_DATA_DIR = os.path.join(BASE_DIR, 'data', 'clean_data')
print(f"Cleaned csv file path: {CLEANED_DATA_DIR}")


Cleaned csv file path: c:\Users\user\Documents\GitHubRepos\DS6103\SD6103_Project\data\clean_data


In [6]:
# checking if the cleaned csv files have the required columns
from pathlib import Path

SOURCES = {
    "town": {
        "file": "hdb_town",
        "columns": {
            "TownName": "town_name",
        },
    },
    "hdb_block": {
        "file": "hdb_block_clean",
        "columns": {
            "town_name":      "town_name",
            "blk_no":         "block_number",
            "street":         "street",
            "year_completed": "year_completed",
            "lat":            "latitude",
            "lng":            "longitude",
            "postal":         "postal",
        },
    },
    "hawker_centre": {
        "file": "hawker_centre",
        "columns": {
            "hawker_name":              "hawker_name",
            "street":                   "street",
            "latitude":                 "latitude",
            "longitude":                "longitude",
            "postal":                   "postal",
            "no_of_mkt_produce_stalls": "no_of_mkt_produce_stalls",
            "no_of_cooked_food_stalls": "no_of_cooked_food_stalls",
        },
    },
    "proximity_level": {
        "file": "proximity_level",
        "columns": {
            "level_id":       "level_id",
            "category":       "category",
            "distance_min_m": "distance_min_m",
            "distance_max_m": "distance_max_m",
        },
    },
    "proximity": {
        "file": "proximity",
        "columns": {
            "blk_no":      "blk_no",
            "street":      "street",
            "hawker_name": "hawker_name",
            "distance_m":  "distance_m",
        },
    },
}

def header_of(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        return next(csv.reader(f))
    
ok = True

for table, cfg in SOURCES.items():
    path = Path(CLEANED_DATA_DIR) / f"{cfg['file']}.csv"
    if not path.exists():
        print(f"MISSING  {path}")
        ok = False
        continue

    have  = header_of(path)
    want  = list(cfg["columns"])
    gap   = [c for c in want if c not in have]
    skip  = [c for c in have if c not in want]

    print(f"{'OK  ' if not gap else 'GAP '} {table:14s} {path.name}")
    print(f"keep columns: {want}")
    # print(f"      keep {len(want)}, ignore {len(skip)}" + (f" -> {skip}" if skip else ""))
    if gap:
        print(f"MISSING   : {gap}")
        print(f"file has  : {have}")
        ok = False

print("\nReady to load." if ok else "\nFix the gaps above.")

OK   town           hdb_town_YM.csv
keep columns: ['TownName']
OK   hdb_block      hdb_block_clean_YM.csv
keep columns: ['town_name', 'blk_no', 'street', 'year_completed', 'lat', 'lng', 'postal']

Ready to load.


In [7]:
# loading the data into staging table
import io

def load_staging(table, cfg, chunksize=500_000):
    path     = Path(CLEANED_DATA_DIR) / f"{cfg['file']}.csv"
    csv_cols = list(cfg["columns"])
    sql_cols = [cfg["columns"][c] for c in csv_cols]

    with engine.begin() as conn:
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS staging"))
        conn.execute(text(f"DROP TABLE IF EXISTS staging.{table}"))
        conn.execute(text(f"CREATE TABLE staging.{table} (" + ", ".join(f'"{c}" TEXT' for c in sql_cols) + ")"))

    copy_sql = (f'COPY staging.{table} ({", ".join(chr(34)+c+chr(34) for c in sql_cols)}) '
                "FROM STDIN WITH (FORMAT csv)")

    n, t0 = 0, time.perf_counter()
    raw = engine.raw_connection()
    try:
        with raw.cursor() as cur:
            for chunk in pd.read_csv(path, usecols=csv_cols, dtype=str, keep_default_na=False, chunksize=chunksize):
                chunk = chunk[csv_cols]
                buf = io.StringIO()
                chunk.to_csv(buf, index=False, header=False)
                buf.seek(0)
                cur.copy_expert(copy_sql, buf)
                n += len(chunk)
        raw.commit()
    finally:
        raw.close()

    print(f"{table:14s} {n:>9,} rows   {time.perf_counter()-t0:5.1f}s")


for table, cfg in SOURCES.items():
    load_staging(table, cfg)

town                  26 rows     0.0s
hdb_block         10,194 rows     0.1s


In [8]:
# Creating and calling stored procedures
import re
import time

SP_DIR = Path(BASE_DIR) / "StoredProcedures"

sp_files = sorted(SP_DIR.glob("sp*.sql"))
print("Run order:", [f.name for f in sp_files])


PROC_NAME = re.compile(r"CREATE\s+(?:OR\s+REPLACE\s+)?PROCEDURE\s+([\w.]+)", re.IGNORECASE)

def deploy_and_call(path):
    sql = path.read_text(encoding="utf-8")

    m = PROC_NAME.search(sql)
    if not m:
        raise ValueError(f"No CREATE PROCEDURE found in {path.name}")
    proc = m.group(1)

    t0 = time.perf_counter()
    raw = engine.raw_connection()
    try:
        with raw.cursor() as cur:
            cur.execute(sql)
            cur.execute(f"CALL {proc}();")
        raw.commit()

        notices = raw.driver_connection.notices
        for n in notices:
            print("   ", n.strip())
        notices.clear()
    except Exception:
        raw.rollback()
        raise
    finally:
        raw.close()

    print(f"OK   {path.name} -> {proc}  ({time.perf_counter() - t0:.1f}s)")


for f in sp_files:
    try:
        deploy_and_call(f)
    except Exception as e:
        print(f"FAIL {f.name}: {e}")
        break


Run order: ['sp01_SchemaCreation.sql', 'sp02_load_town.sql', 'sp03_load_hdb_block.sql']
    NOTICE:  relation "town" already exists, skipping
    NOTICE:  town schema created
    NOTICE:  relation "hdb_block" already exists, skipping
    NOTICE:  hdb_block schema created
    NOTICE:  relation "idx_hdb_block_town" already exists, skipping
    NOTICE:  indexes created
    NOTICE:  --------------------------------------------
    NOTICE:  schema built in 5ms
OK   sp01_SchemaCreation.sql -> public.sp_build_schema  (0.1s)
    NOTICE:  Committed. Towns inserted: 0
OK   sp02_load_town.sql -> public.sp_load_town  (0.0s)
    NOTICE:  Committed. Rows inserted: 0
OK   sp03_load_hdb_block.sql -> public.sp_load_hdb_block  (0.0s)
